# **HW02 - EDA cycle.**

## Задача: найти датасет *(много данных, много категорий, есть пропуски)* и провести EDA.
- Сбор.
- Очистка.
- Исследование.
- Предобработка.
- Гипотезы и выводы.

## 1) Выбор датасета
В качестве датасета я выбрал **Spotify Tracks Dataset** с kaggle *(https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)*  
**Сам датасет достаточно хороший, в нем нет пропусков, поэтому я исскуственно создам пропуски.**
### Пару слов о полях датасете (кроме очевидных параметров):
- popularity - популярность песни от 0 до 100, где 100 - очень популярная. 
- explicit - откровенные тексты (я не знаю как это нормально и правильно перевести).
- danceability - от 0 до 1 насколько песня танцевальная.
- energy - здесь понимается скорость, громкость, энергия песни. од 0 до 1 (спокойная, энергичная).
- key - тональность песни. 0 - C, 1 - C#, и т.д.
- loudness - громкость песни в децибеллах.
- mode - минорная (0) или мажорная (1) тональность.
- speechiness - показывает как много текста в песне. от 0 до 1. 1 - подкаст, 0 - мелодия, выше 0.66 - песня где много слов, 0.33 - 0.66, как правило обычная песня, например рэп, ниже 0.33 - песня, где мало слов.
- acousticness - показывает вероятность, с которой песня аккустическая.
- instrumentalness - показывает вероятность, с которой песня почти без вокала.
- liveness - показывает вероятность, с которой песня была записана в лайве.
- valence - показывает вероятность, с которой песня позитивная.
- time_signature - ритм песни (цифра / 4).  
*Остальное по идее понятно*.

## 2) Быстрый обзор данных
### 2.1) Загрузка датасета и его порча.
Еще я переведу миллисекунды в минуты, название также поменяю.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

df = pd.read_csv("spotify-tracks-dataset.csv", index_col = 0) #вручную изменил название первого столбца (track_number), т.к. оно было пустое
pd.set_option('display.max_columns', None) #почему-то без этого он не выводит поле key

np.random.seed(42)

num_deletion = 10000

df['explicit'] = df['explicit'].astype(int) #меняем bool на float, чтобы можно было занулить некоторые элементы из этого стоблца

for idx in range (num_deletion):
    row_idx = np.random.randint(0, len(df))
    col_idx = np.random.randint(0, len(df.columns))
    df.iloc[row_idx, col_idx] = np.nan #вот из-за того, что мы присваиваем nan, у нас все int превратятся во float
df['duration_ms'] = df['duration_ms'] / 60000
df = df.rename(columns={'duration_ms': 'duration_min'})


### 2.2) df.head(), df.tail(), df.shape.

In [ ]:
display(df.head())
display(df.tail())
print("Dataset shape: ", df.shape)

### 2.3) df.info(), df.describe(), df.describe(include = "str").

In [ ]:
df.info()
display(df.describe())
display(df.describe(include = "str"))

### 2.4) Проверки качества: isnull().sum(), поиск дубликатов, проверка адекватности типов данных.

In [ ]:
print(df.isnull().sum())
print("Duplicates: ", df.duplicated().sum())
print(df.dtypes)

### 2.5) Мини-вывод по датасету.
В датасете 114000 записей, 20 разных параметров, около 400 дубликатов. Исскуственно испорчены некоторые записи, поэтому будет хорошо учиться на таком датасете.

## 3) Пропуски и очистка.
### 3.1) Dropna(), fillna(). 
- Если нет параметра типа string, то я просто дропну эту строку, т.к. невозможно задать стандартное, среднее значение жанру или популярное значение жанру, названию или исполнителю. Популярное тоже не возьмешь, т.к. в датасете не собраны песни одного жанра либо автора. Эти параметры достаточно важные, они будут учитываться, но т.к. мы их заполнить не можем, мы их просто дропнем.  
- Все float64, мы заполним просто средними значениями. Причина та же - непонятно, какое значение выбрать, поэтому чтобы не ломать статистику возьмем везде среднее, даже там, где только 0 и 1.  
- В конце концов пустых ячеек не так много, и чем больше мы их сохраним, тем лучше.  
- В итоге мы удалили около 2500 ячеек, и 7500 восстановили. Это также проверяется тем, что у нас 5 полей типа string, количество пропусков мы можем посчитать, взяв данные из таблицы.
- Заполнение модой я не хотел использовать, т.к. оно немного ломает статистику, поэтому лучше оставить как есть и анализировать другие параметры.
- В конце снова выведены данные, чтобы посмотреть на количество пропусков.

In [ ]:
float_columns = df.select_dtypes(include = ['float64']).columns
df[float_columns] = df[float_columns].fillna(df[float_columns].mean())

str_columns = df.select_dtypes(include = ['str']).columns
df = df.dropna(subset = str_columns)

df.info()
print(df.isnull().sum())

### 3.2) Как видим, всё прекрасно починено.

## 4) Расширенная статистика
### 4.1) Для числовых колонок выведем: min, max, mean, median, mode.

In [ ]:
print("Min:")
print(df.min(numeric_only = True))
print()
print("Max:")
print(df.max(numeric_only = True))
print()
print("Mean:")
print(df.mean(numeric_only = True))
print()
print("Median:")
print(df.median(numeric_only = True))
print()
print("Mode:")
print(df.mode(numeric_only = True).iloc[0])
print()

### 4.2) Посчитаем percentile/quantile (например, 5, 25, 50, 75, 95 перцентили).

In [ ]:
df